# Notebook 4: Constructing Voronoi Polygons

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/1854-Cholera-Outbreak-London-Advanced-2/blob/d1b56c135b0fab2f470b837a51e6777dfc20b286/NB03-Constructing-Voronoi-Polygons.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB03-Constructing-Voronoi-Polygons.ipynb` at `d1b56c135b0fab2f470b837a51e6777dfc20b286`; original SHA-256 `536a3ef124229ccc6cda3d51ff9caed6a680816dfc0fc2b2568a79db8b474944`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run resources/library.py

# Notebook 4: Constructing Voronoi Polygons

## The `voronoi_finite_polygons_2d` function

The code below is a function to create 2d voronoi polygons given seed points. This code augments the native `Voronoi()` function in the Scipy Spatial package by closing infinite vertices.

In this case, we use the pumps as seed points.

In [ ]:
%%writefile resources/voronoi2.py
# https://gist.github.com/pv/8036995
import numpy as np

def voronoi_finite_polygons_2d(vor, radius=None):
    """
    Reconstruct infinite voronoi regions in a 2D diagram to finite
    regions.
    Parameters
    ----------
    vor : Voronoi
        Input diagram
    radius : float, optional
        Distance to 'points at infinity'.
    Returns
    -------
    regions : list of tuples
        Indices of vertices in each revised Voronoi regions.
    vertices : list of tuples
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    """

    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = np.ptp(vor.points).max()*2

    # Construct a map containing all ridges for a given point
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    # Reconstruct infinite regions
    for p1, region in enumerate(vor.point_region):
        vertices = vor.regions[region]

        if all(v >= 0 for v in vertices):
            # finite region
            new_regions.append(vertices)
            continue

        # reconstruct a non-finite region
        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                # finite ridge: already in the region
                continue

            # Compute the missing endpoint of an infinite ridge

            t = vor.points[p2] - vor.points[p1] # tangent
            t /= np.linalg.norm(t)
            n = np.array([-t[1], t[0]])  # normal

            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius

            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())

        # sort region counterclockwise
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(angles)]

        # finish
        new_regions.append(new_region.tolist())

    return new_regions, np.asarray(new_vertices)

## Load Cholera data

In [ ]:
import pandas as pd

In [ ]:
pumps_df = pd.read_csv('data/pumps.csv')

pumps_df

In [ ]:
deaths_df = pd.read_csv('data/cholera_deaths.csv')

deaths_df.head()

In [ ]:
import numpy as np

## Create points lists (arrays) out of the Cholera data

In [ ]:
points_pumps = np.array([[lon,lat] for lat,lon in pumps_df[['LAT','LON']].values])

points_pumps

In [ ]:
points_deaths = np.array([[lon,lat] for lat,lon in deaths_df[['LAT','LON']].values])

points_deaths

## Create Geopandas geodataframes out of pumps and deaths data

In [ ]:
from shapely.geometry import Polygon, Point, MultiPoint
import geopandas as gpd

In [ ]:
crs = 'EPSG:4326'
pumps_geometry = [Point(xy) for xy in zip(pumps_df.LON,pumps_df.LAT)]
pumps_gdf = gpd.GeoDataFrame(pumps_df.drop(['LON', 'LAT'], axis=1), crs=crs, geometry=pumps_geometry)

pumps_gdf.plot()

In [ ]:
pumps_gdf

### Save the geodataframes to Shapefile and JSON format

In [ ]:
pumps_gdf.to_file('outputs/pumps.shp')
pumps_gdf.to_file("outputs/pumps.json", driver="GeoJSON")

### Quick plots of geodataframes

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
crs = 'EPSG:4326'
deaths_geometry = [Point(xy) for xy in zip(deaths_df.LON, deaths_df.LAT)]
deaths_gdf = gpd.GeoDataFrame(deaths_df.drop(['LON', 'LAT'], axis=1), \
                              crs=crs, geometry=deaths_geometry)

deaths_gdf.plot()

In [ ]:
pumps_gdf.plot()

### Save the geodataframes to Shapefile and JSON format

In [ ]:
deaths_gdf.to_file('outputs/deaths.shp')
deaths_gdf.to_file("outputs/deaths.json", driver="GeoJSON")

## Compute Voronoi polygons

In [ ]:
from scipy.spatial import Voronoi

Let's run the voronoi 2d generation Python file that we saved earlier.

In [ ]:
%run resources/voronoi2.py

In [ ]:
# compute Voronoi tesselation using scipy spatial Voronoi function
# note: this function does not create regions (polygons) for infinite vertices
vor = Voronoi(points_pumps)

vor

In [ ]:


# compute 2d voronoi with finite polygons
regions, vertices = voronoi_finite_polygons_2d(vor)

# display regions (polygon indices to vertices)
regions

In [ ]:
# display voronoi polygon vertices (all of them)
# note: these have to be used with regions where the indices point to each vertex in this list
vertices

In [ ]:
# voronoi input points (pump locations) from the original scipy spatial Voronoi function
vor.points

## Store Voronoi polygons in a Geopandas geodataframe

In [ ]:
# create an empty geodataframe
# Don't add crs to an empty geodataframe, not allowed by geopandas!
vor_poly_gdf1 = gpd.GeoDataFrame()

vor_poly_gdf1

That's what an empty geodataframe looks like!

In [ ]:
# initialize index
index = 0

for region in regions:
    
    # print vertices for each region (voronoi polygon)
    #print(vertices[region])
    
    # convert each polygon into Shapely Polygon
    vor_poly = Polygon(vertices[region])
    
    # save Polygon as geopandas geometry column
    vor_poly_gdf1.loc[index, 'geometry'] = vor_poly
    
    # display corresponding vor input point (pump location: lat, lon)
    vorpoint_lat = vor.points[index][1]
    vorpoint_lon = vor.points[index][0]
    #print("vor.point:", vorpoint_lat, vorpoint_lon)
    
    # save lat, lon to geopandas geodataframe
    vor_poly_gdf1.loc[index, 'vorpoint_lat'] = vorpoint_lat
    vor_poly_gdf1.loc[index, 'vorpoint_lon'] = vorpoint_lon

    # increment index by 1
    index = index + 1

In [ ]:
# now we can add crs, to the geodataframe
vor_poly_gdf1.crs = 'EPSG:4326'

In [ ]:
vor_poly_gdf1

## Quick plot of the Voronoi polygons using Geopandas

In [ ]:
vor_poly_gdf1.plot(alpha=0.2, edgecolor='black', linewidth=2)

Note how the latitude and longitude values would mean a folium plot of the voronoi cells would extend far beyond the viewing window of a web map.

## Geopandas spatial join - pumps and Voronoi polygons

In [ ]:
# do a geopandas spatial join (intersection)
# this assumes pump locations in geometry column are within (intersects with) voronoi polygon
vor_poly_gdf2 = \
    gpd.sjoin(vor_poly_gdf1, pumps_gdf, how="inner", predicate='intersects')\
    .drop(columns=['index_right'])

vor_poly_gdf2

Getting warmer...

## Clipping the Voronoi polygons

The polygons above will extend beyond screen if plotted on a `folium` map. We will clip the voronoi polygons that extend beyond the convex hull that we created earlier in Notebook 2. Code follows below.

### Retrieve the convex hull polygon we created earlier in Notebook 2

In [ ]:
convexhull_gdf = gpd.read_file('outputs/convexhull.shp')

### Use Geopandas overlay to obtain the intersection of Voronoi polygons with the convex hull polygon [1]

In [ ]:
clipped_voronoi_gdf = gpd.overlay(
    convexhull_gdf,
    vor_poly_gdf2,
    how="intersection"
).drop(columns=['FID_1']).rename(columns={'FID_2':'FID'})

clipped_voronoi_gdf

### Quick plot of the clipped Voronoi polygons

In [ ]:
clipped_voronoi_gdf.plot(alpha=0.2, edgecolor='black',linewidth=2)

That looks very promising...

Now, let's put the two plots together - the unclipped voronoi cells and the clipped ones. Here's a simple `matplotlib` plotting lesson below. We defined `matplotlib.pyplot` earlier as `plt`.

In [ ]:
# note how we create subplots side by side using subplot parameters, nrows and ncols
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 13))

# here are the two plots
# since we used two columns, column 1 uses axes[0] and column 2 uses axes[1]
vor_poly_gdf1.plot(alpha=0.2, edgecolor='black',linewidth=2, \
                   ax=axes[0])
clipped_voronoi_gdf.plot(alpha=0.2, edgecolor='black',linewidth=2, 
                         ax=axes[1], color='red')

# let's put the titles so we will know what we are looking at
fig.suptitle("Unclipped (left) and Clipped (right) Voronoi Cells", fontsize=14)
axes[0].set_title("Unclipped (very large polygon)")
axes[1].set_title("Clipped (very small polygon)")

fig.tight_layout()

Note that the two figures are not the same size and scale.

### Save the clipped Voronoi polygons

In [ ]:
clipped_voronoi_gdf.to_file('outputs/clipped_voronoi.shp')
clipped_voronoi_gdf.to_file("outputs/clipped_voronoi.json", driver="GeoJSON")

In [ ]:
clipped_voronoi_gdf

## Counting Death Locations Within Voronoi Polygons [2]

One of the most important things to study in this case is to determine which Voronoi polygon contains the most number of death locations. 

In [ ]:
# Read the data.
polygons = gpd.GeoDataFrame.from_file('outputs/clipped_voronoi.shp')
points = gpd.GeoDataFrame.from_file('outputs/deaths.shp')

Let's write the function, `points_in_polygons` below to a Python file as we will use this again in Notebook 6.

In [ ]:
%%writefile resources/points_in_polygons.py
import geopandas as gpd
import pandas as pd

def points_in_polygons(points_gdf, polygons_gdf):
    
    polygons = polygons_gdf
    points = points_gdf
    
    # Make a copy because I'm going to drop points as I
    # assign them to polys, to speed up subsequent search.
    pts = points.copy() 

    # We're going to keep a list of how many points we find.
    pts_in_polys = []

    # Loop over polygons with index i.
    for i, poly in polygons.iterrows():

        # Keep a list of points in this poly
        pts_in_this_poly = []

        # Now loop over all points with index j.
        for j, pt in pts.iterrows():
            if poly.geometry.contains(pt.geometry):
                # Then it's a hit! Add it to the list,
                # and drop it so we have less hunting.
                pts_in_this_poly.append(pt.geometry)
                pts = pts.drop([j])

        # We could do all sorts, like grab a property of the
        # points, but let's just append the number of them.
        pts_in_polys.append(len(pts_in_this_poly))

    # Add the number of points for each poly to the dataframe.
    polygons['points_in_polygon'] = pd.Series(pts_in_polys)
    
    return polygons

In [ ]:
%run resources/points_in_polygons.py

In [ ]:
polygons = points_in_polygons(points, polygons)

polygons

### Let's fix the column names  of the geodataframe holding the Voronoi cells with counts

In [ ]:
clipped_voronoi_counts_gdf = \
    polygons.rename(\
            columns={'vorpoint_l':'vorpoint_lat',\
                     'vorpoint_1':'vorpoint_lon',\
                     'points_in_polygon':'points_in_voronoi'
                    })

clipped_voronoi_counts_gdf

### Save the clipped Voronoi polygons with death location counts

In [ ]:
clipped_voronoi_counts_gdf.to_file('outputs/clipped_voronoi_counts.shp')
clipped_voronoi_counts_gdf.to_file("outputs/clipped_voronoi_counts.json", driver="GeoJSON")

## And now, for amplified cognition - Draw a simple horizontal bar chart

### Horizontal Bar Showing Death Locations by Voronoi Cells (Pump Catchment)

In [ ]:
ax = pd.DataFrame(clipped_voronoi_counts_gdf)\
    .plot.barh(y='points_in_voronoi', x='Street', color='k', figsize=(8,5))
ax.set_title('Death Locations in Voronoi Cells of Pumps (by Street Location)', fontsize=16)
ax.tick_params(axis = 'both', which = 'major', labelsize = 14)
ax.set_ylabel('Voronoi Cells by Pump', fontsize=16)
ax.set_xlabel('Death Location Count', fontsize=16)

Which street pump catchment area (Voronoi cell) has the most number of mortality locations?

### Draft Map Overlaying Pumps, Death Locations and Voronoi Cells

In [ ]:
width = 13
height = 13

fig, ax = plt.subplots(figsize=(width, height))

# putting together the three plots
clipped_voronoi_gdf.plot(ax=ax, alpha=0.1, edgecolor='black',linewidth=2)
deaths_gdf.plot(ax=ax, alpha=0.2, color='darkblue')
pumps_gdf.plot(ax=ax,color='red')

ax.tick_params(axis = 'both', which = 'major', labelsize = 14)
# always good to remember, x maps to longitude, y maps to latitude 
ax.set_ylabel('Latitude', fontsize=16)
ax.set_xlabel('Longitude', fontsize=16)
ax.set_title('Deaths Locations within Pump-derived Voronoi Cells, Soho District', \
             fontsize=16)

## Reflection Questions

1. What did you learn from going through the Voronoi production code from Notebooks 2-4?
2. What else can we do to improve our Voronoi production process?

## References

1. How to use geopandas' overlay method to find the area of intersections between two datasets: https://github.com/datadesk/geopandas-intersection-area-example
2. Count number of points in multipolygon shapefile using Python: https://stackoverflow.com/questions/27606924/count-number-of-points-in-multipolygon-shapefile-using-python